In [1]:
import os
import sys
sys.path.append("../../../../")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import copy
import torch
from datetime import datetime
from utils.helper import ModelConfig, color_print
from utils.dataset_utils.load_dataset import (
    load_data, Embedding
)
from utils.model_utils.save_module import save_module
from utils.model_utils.load_model import load_model
from utils.model_utils.evaluate import evaluate_model, get_sparsity, similar
from utils.dataset_utils.perturb import make_example
from utils.dataset_utils.sampling import SamplingDataset
from utils.prune_utils.prune_head import (
    compute_heads_importance,
    head_importance_prunning,
)
from utils.prune_utils.prune import recover_tangling_identification, prune_concern_identification, prune_wanda

In [3]:
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
from torch.nn import CrossEntropyLoss, MSELoss
from functools import partial
import torch.nn.functional as F
import math
from sklearn.metrics import classification_report
import gc

In [4]:
name= "YahooAnswersTopics"
device = torch.device("cuda:0")
checkpoint = None
batch_size=16
num_workers=4
seed=44

In [5]:
model_config = ModelConfig(name, device)
num_labels = model_config.config["num_labels"]
model, tokenizer, checkpoint = load_model(model_config)

Loading the model.
{'model_name': 'fabriceyhc/bert-base-uncased-yahoo_answers_topics', 'task_type': 'classification', 'architectures': 'bert', 'dataset_name': 'YahooAnswersTopics', 'num_labels': 10, 'cache_dir': 'Models'}
The model fabriceyhc/bert-base-uncased-yahoo_answers_topics is loaded.


The model fabriceyhc/bert-base-uncased-yahoo_answers_topics is loaded.

In [6]:
train_dataloader, valid_dataloader, test_dataloader = load_data(
    name, batch_size=batch_size, num_workers=num_workers, do_cache=True, seed=seed
)

Loading cached dataset YahooAnswersTopics.
The dataset YahooAnswersTopics is loaded
{'dataset_name': 'YahooAnswersTopics', 'path': 'yahoo_answers_topics', 'config_name': 'yahoo_answers_topics', 'text_column': 'question_title', 'label_column': 'topic', 'cache_dir': 'Datasets/Yahoo', 'task_type': 'classification'}


{'dataset_name': 'YahooAnswersTopics', 'path': 'yahoo_answers_topics', 'config_name': 'yahoo_answers_topics', 'text_column': 'question_title', 'label_column': 'topic', 'cache_dir': 'Datasets/Yahoo', 'task_type': 'classification'}

In [7]:
positive_embeddings, negative_embeddings = make_example(
    model, model_config, data_loader=valid_dataloader,
    example_num=3000, emb_num=1, class_num=10, true_ratio=0.5,
    step_epsilon=0.01, max_epsilon=10.0
)

positive num :  1500
negative num :  1500


Calculating all epsilons:   0%|                          | 0/10 [00:00<?, ?it/s]

per_class_positive_example_num :  150
per_class_negative_example_num :  150


150

per_class_negative_example_num : 

150

In [8]:
pos_dataloader = DataLoader(positive_embeddings, batch_size=4, shuffle=True, num_workers=16)
neg_dataloader = DataLoader(negative_embeddings, batch_size=4, shuffle=True, num_workers=16)

In [9]:
batch_size = 16
num_workers = 4
num_samples = 128
ci_ratio = 0.6
seed = 44
include_layers = ["attention", "intermediate", "output"]
exclude_layers = None

In [10]:
for concern in range(num_labels):
    print(f"-------------------{concern}----------------------")
    module = copy.deepcopy(model)
    pos = copy.deepcopy(pos_dataloader)
    neg = copy.deepcopy(neg_dataloader)
    valid = copy.deepcopy(valid_dataloader)

    positive_examples = SamplingDataset(
        pos, concern, num_samples, num_labels, True, 4, device=device, resample=False, seed=seed
    )
    negative_examples = SamplingDataset(
        neg, concern, num_samples, num_labels, True, 4, device=device, resample=False, seed=seed
    )
    prune_concern_identification(
        module,
        model_config,
        positive_examples,
        negative_examples,
        include_layers=include_layers,
        exclude_layers=exclude_layers,
        sparsity_ratio=ci_ratio,
    )

    print(f"Evaluate the pruned model {concern}")
    result = evaluate_model(module, model_config, test_dataloader)
    get_sparsity(module)

    similar(model, module, valid, concern, num_samples, num_labels, device=device, seed=seed)

-------------------0----------------------
Evaluate the pruned model 0


Evaluating the model:   0%|                            | 0/1875 [00:00<?, ?it/s]

Loss: 1.5445
Precision: 0.6444, Recall: 0.5471, F1-Score: 0.5501
              precision    recall  f1-score   support

           0       0.54      0.51      0.52      2941
           1       0.84      0.23      0.36      2997
           2       0.84      0.45      0.58      3016
           3       0.54      0.37      0.44      2978
           4       0.75      0.75      0.75      3017
           5       0.96      0.58      0.72      3004
           6       0.31      0.44      0.36      3037
           7       0.36      0.77      0.49      3026
           8       0.49      0.82      0.61      2997
           9       0.82      0.55      0.66      2987

    accuracy                           0.55     30000
   macro avg       0.64      0.55      0.55     30000
weighted avg       0.64      0.55      0.55     30000

adding eps to diagonal and taking inverse
taking square root
dot products...
trying to take final svd
computed everything!
adding eps to diagonal and taking inverse
taking squa

Evaluating the model:   0%|                            | 0/1875 [00:00<?, ?it/s]

Loss: 1.5765
Precision: 0.6469, Recall: 0.5475, F1-Score: 0.5548
              precision    recall  f1-score   support

           0       0.47      0.51      0.49      2941
           1       0.80      0.40      0.53      2997
           2       0.81      0.49      0.61      3016
           3       0.53      0.32      0.40      2978
           4       0.73      0.76      0.75      3017
           5       0.97      0.55      0.70      3004
           6       0.55      0.31      0.40      3037
           7       0.30      0.82      0.44      3026
           8       0.49      0.81      0.61      2997
           9       0.82      0.51      0.63      2987

    accuracy                           0.55     30000
   macro avg       0.65      0.55      0.55     30000
weighted avg       0.65      0.55      0.55     30000

adding eps to diagonal and taking inverse
taking square root
dot products...
trying to take final svd
computed everything!
adding eps to diagonal and taking inverse
taking squa

...

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           9       1.00      0.95      0.97       128

    accuracy                           0.95       128
   macro avg       0.50      0.47      0.49       128
weighted avg       1.00      0.95      0.97       128


Evaluating the model: 0it [00:00, ?it/s]

Loss: 3.0527

Precision: 0.0000, Recall: 0.0000, F1-Score: 0.0000

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       0.0
           5       0.00      0.00      0.00       0.0
           7       0.00      0.00      0.00       0.0
           8       0.00      0.00      0.00       0.0
           9       0.00      0.00      0.00     128.0

    accuracy                           0.00     128.0
   macro avg       0.00      0.00      0.00     128.0
weighted avg       0.00      0.00      0.00     128.0
